<a href="https://colab.research.google.com/github/vinhlq2512/research-code-lab/blob/feat/tapta-data-scalability/notebooks/tapta_p5_ablation_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🚀 TAPTA P5: Contrastive-Margin Tree & Anisotropy-Resistant Router
### Đề tài: Continual Relation Extraction with Task-Aware Prompt Adaptation (Thạc sĩ)

Notebook này triển khai **Toàn diện Bộ Thực Nghiệm Ablation P5** nhằm kiểm chứng sự vượt trội so với baseline P4:
- **P4 Baseline:** Mô hình đối chứng P4 chuẩn tái lập (Centroid Router + Decision Soft Fusion)
- **P5-A:** + Module A (Anisotropy-Resistant Task Router với Subcentroid Ensemble)
- **P5-B:** + Module A + Module C (Confidence-Gated Candidate Masking loại bỏ độc tính Soft Fusion)
- **P5-C:** + Module A + Module B + Module C (Supervised Prototype Contrastive Loss mở rộng Margin)
- **P5-Full:** Full P5 (+ Module D: Momentum Root Prompt Anchor Regularization chống lãng quên)

> **Ràng buộc nghiên cứu:** Strict Equal 5-shot, $M=0$ Rehearsal-Free, 100% Frozen BERT-base 109.5M, Task-Agnostic Class-Incremental (CIL).

## 1. Kiểm tra Môi trường GPU (NVIDIA CUDA Preflight Check)

In [ ]:
import torch
print("=" * 60)
print("CUDA / GPU PREFLIGHT CHECK")
print("=" * 60)
print(f"CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Device:     {torch.cuda.get_device_name(0)}")
    print(f"Total VRAM:     {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")
    print(f"CUDA Version:   {torch.version.cuda}")
else:
    print("⚠️ CẢNH BÁO: Không tìm thấy GPU CUDA! Vui lòng vào Runtime -> Change runtime type -> Chọn T4 GPU.")
print("=" * 60)

## 2. Clone / Cập Nhật Repository `research-code-lab`
Tự động tải nhánh `feat/tapta-data-scalability` chứa toàn bộ mã nguồn P5 và cài đặt dependencies.

In [ ]:
import os
REPO_URL = "https://github.com/vinhlq2512/research-code-lab.git"
BRANCH = "feat/tapta-data-scalability"
REPO_DIR = "/content/research-code-lab"

if not os.path.exists(REPO_DIR):
    !git clone -b {BRANCH} {REPO_URL} {REPO_DIR}
else:
    !git -C {REPO_DIR} fetch origin
    !git -C {REPO_DIR} checkout {BRANCH}
    !git -C {REPO_DIR} pull origin {BRANCH}

%cd {REPO_DIR}
!pip install -q transformers pyyaml scikit-learn tabulate pytest

## 3. Chạy Kiểm Thử Tự Động (Unit Tests Verification)
Xác thực toàn bộ modules toán học của P5 trước khi chạy thực nghiệm.

In [ ]:
!pytest experiments/tests/test_p5_modules.py

## 4. Thực Nghiệm Ablation Tuần Tự P5 Trên TACRED Track A
*(41 quan hệ, 8 tác vụ, strict 5-shot, Rehearsal-Free $M=0$, seed 2021)*

Chạy tuần tự 5 cấu hình: P4 Baseline, P5-A, P5-B, P5-C, P5-Full.

In [ ]:
# Bước 0: P4 Baseline (Mô hình đối chứng chuẩn)
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_tacred_5shot_seed2021.yaml \
    --mode p4_baseline \
    --device cuda

# Bước 1: P5-A (Tuned Subcentroid Task Router)
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_tacred_5shot_seed2021.yaml \
    --mode p5_a \
    --device cuda

# Bước 2: P5-B (+ Confidence-Gated Candidate Masking)
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_tacred_5shot_seed2021.yaml \
    --mode p5_b \
    --device cuda

# Bước 3: P5-C (+ Supervised Prototype Contrastive Loss)
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_tacred_5shot_seed2021.yaml \
    --mode p5_c \
    --device cuda

# Bước 4: P5-Full (Full P5 + Momentum Root Prompt Anchor Regularization)
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_tacred_5shot_seed2021.yaml \
    --mode p5_full \
    --device cuda

## 5. Thực Nghiệm Trên Benchmark Chính FewRel Track A
*(80 quan hệ, 8 tác vụ, strict 5-shot, Rehearsal-Free $M=0$, seed 2021)*

Chạy đối chứng P4 Baseline và P5-Full trên FewRel Track A.

In [ ]:
# Bước 0: P4 Baseline trên FewRel
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_fewrel_5shot_seed2021.yaml \
    --mode p4_baseline \
    --device cuda

# Bước 1: P5-Full trên FewRel
!python experiments/run_p5.py \
    --config configs/p5_candidates/p5_fewrel_5shot_seed2021.yaml \
    --mode p5_full \
    --device cuda

## 6. Tổng Hợp Bảng So Sánh Khoa Học & Tự Động Tải Về Artifacts
Đọc toàn bộ file `summary.json`, in bảng ma trận so sánh các cấu hình ablation và đóng gói file zip kết quả.

In [ ]:
import os
import json
import shutil
from pathlib import Path
from tabulate import tabulate
from google.colab import files

print("=" * 80)
print("BẢNG TỔNG HỢP KẾT QUẢ ABLATION P5 (TACRED & FEWREL)")
print("=" * 80)

runs_dir = Path("/content/research-code-lab/runs")
rows = []

for summary_file in sorted(runs_dir.glob("p5_*/tacred_5shot/seed_2021/summary.json")):
    try:
        with open(summary_file) as f:
            d = json.load(f)
        mode = d.get("mode", summary_file.parent.parent.name)
        a_last = d.get("final_average_accuracy", 0) * 100
        aia = d.get("average_incremental_accuracy", 0) * 100
        f_rate = d.get("average_forgetting", 0) * 100
        r_top1 = d.get("final_router_top1_accuracy", 0) * 100
        r_top2 = d.get("final_router_top2_coverage", 0) * 100
        rows.append(["TACRED", mode.upper(), f"{a_last:5.2f}%", f"{aia:5.2f}%", f"{f_rate:5.2f}%", f"{r_top1:5.2f}%", f"{r_top2:5.2f}%"])
    except Exception as e:
        pass

for summary_file in sorted(runs_dir.glob("p5_*/fewrel_5shot/seed_2021/summary.json")):
    try:
        with open(summary_file) as f:
            d = json.load(f)
        mode = d.get("mode", summary_file.parent.parent.name)
        a_last = d.get("final_average_accuracy", 0) * 100
        aia = d.get("average_incremental_accuracy", 0) * 100
        f_rate = d.get("average_forgetting", 0) * 100
        r_top1 = d.get("final_router_top1_accuracy", 0) * 100
        r_top2 = d.get("final_router_top2_coverage", 0) * 100
        rows.append(["FEWREL", mode.upper(), f"{a_last:5.2f}%", f"{aia:5.2f}%", f"{f_rate:5.2f}%", f"{r_top1:5.2f}%", f"{r_top2:5.2f}%"])
    except Exception as e:
        pass

headers = ["Dataset", "Configuration", "Final Acc (A_last)", "AIA", "Forgetting (F)", "Router Top-1", "Router Top-2"]
print(tabulate(rows, headers=headers, tablefmt="grid"))

# Đóng gói file zip
print("\nĐang đóng gói kết quả thực nghiệm P5 (loại bỏ các file checkpoint quá nặng để tải siêu tốc)...")
export_dir = Path("/content/p5_results_bundle")
if export_dir.exists():
    shutil.rmtree(export_dir)
export_dir.mkdir(parents=True, exist_ok=True)

# Copy toàn bộ runs p5_*
for p5_run in runs_dir.glob("p5_*"):
    dest = export_dir / p5_run.name
    shutil.copytree(p5_run, dest, ignore=shutil.ignore_patterns("*.pt"))

zip_path = "/content/tapta_p5_ablation_results"
shutil.make_archive(zip_path, "zip", export_dir)
file_size_kb = os.path.getsize(zip_path + ".zip") / 1024
print(f"Tệp nén đã tạo: {zip_path}.zip (Kích thước: {file_size_kb:.1f} KB)")
print("Tự động kích hoạt tải xuống...")
files.download(zip_path + ".zip")